# Session 1B — The Vanilla Agent (ReAct Loop)

**GOAL:** Build a REAL AI Agent from scratch.  
No LangChain, no LangGraph — just a Python while-loop that lets the LLM call actual Python functions.

The ReAct pattern: **Reason → Act → Observe → Repeat**

In [ ]:
import os, sys, json, re, warnings, logging

warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

import google.generativeai as genai
from utils.gmail_utils import fetch_recent_emails
from utils.calendar_utils import create_calendar_event
from utils.tools import format_emails

genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
print('Setup complete.')

### Step 1: Define the Tools (Real Python Functions)

These are the functions the agent can call. We intentionally avoid LangChain here to teach the raw mechanics.

In [ ]:
def tool_fetch_emails(**kwargs):
    """Fetch recent emails from Gmail and return them as formatted text."""
    limit = int(kwargs.get('limit', 5))
    emails = fetch_recent_emails(limit=limit)
    return format_emails(emails)

def tool_schedule_meeting(**kwargs):
    """Schedule a meeting on Google Calendar with attendees and a Meet link."""
    time = kwargs.get('time', '')
    attendees_raw = kwargs.get('attendees', [])
    title = kwargs.get('title', 'AI Scheduled Meeting')
    if isinstance(attendees_raw, str):
        attendees = [a.strip() for a in attendees_raw.split(',') if a.strip()]
    else:
        attendees = list(attendees_raw)
    try:
        link = create_calendar_event(time, attendees, title)
        if 'already scheduled' in link:
            return link
        return f"Meeting '{title}' scheduled at {time}. Calendar link: {link}"
    except ValueError as e:
        return f"Error: {str(e)}"

TOOLS = {
    'fetch_emails':     {'function': tool_fetch_emails},
    'schedule_meeting': {'function': tool_schedule_meeting},
}
print('Tools registered:', list(TOOLS.keys()))

### Step 2: System Prompt & Conversation Init

We tell the LLM exactly how to behave: output JSON to call tools, output `{"tool": "DONE"}` when finished.

In [ ]:
SYSTEM_PROMPT = """You are an AI email assistant.
You have access to these tools:

1. fetch_emails(limit)
2. schedule_meeting(time, attendees, title)
   time must be in 'YYYY-MM-DD HH:MM' format.

RULES:
To call a tool, output EXACTLY ONE JSON object:
  {"tool": "tool_name", "args": {"arg1": "value1"}}
When done, output:
  {"tool": "DONE", "summary": "Your final summary here"}
NEVER include markdown fences or anything outside the JSON."""

def extract_json_from_response(text):
    text = text.strip()
    if text.startswith('```'):
        lines = text.split('\n')
        lines = [l for l in lines if not l.strip().startswith('```')]
        text = '\n'.join(lines).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return None

model = genai.GenerativeModel('gemini-flash-latest')
conversation = [
    {'role': 'user',  'parts': [SYSTEM_PROMPT]},
    {'role': 'model', 'parts': ['{"tool": "acknowledge", "status": "ready"}']},
    {'role': 'user',  'parts': ['Please fetch my recent emails, analyze them, and if you find any meeting requests, schedule them on my calendar.']},
]
print('Conversation initialised.')

### Step 3: Run the ReAct Agent Loop

This is the core of every AI agent: **Reason → Act → Observe → Repeat**

In [ ]:
print('Starting the ReAct loop ...\n')

MAX_ITERATIONS = 10
for i in range(1, MAX_ITERATIONS + 1):
    print(f'--- Iteration {i} {"---" * 12}')

    response = model.generate_content(conversation)
    raw = response.text.strip()
    print(f'   LLM says: {raw[:200]}')

    action = extract_json_from_response(raw)

    if action is None:
        print('   Warning: Non-JSON response. Nudging LLM ...')
        conversation.append({'role': 'model', 'parts': [raw]})
        conversation.append({'role': 'user',  'parts': ['Please respond with ONLY a JSON object. No other text.']})
        continue

    tool_name = action.get('tool', '')

    if tool_name == 'DONE':
        print(f'\nFINISHED!')
        print(f'   Summary: {action.get("summary", "No summary.")}')
        break

    if tool_name in TOOLS:
        args = action.get('args', {})
        print(f'   Calling tool: {tool_name}({args})')
        try:
            result = TOOLS[tool_name]['function'](**args)
            print(f'   Result: {str(result)[:300]}')
        except Exception as e:
            result = f'ERROR: {e}'
            print(f'   {result}')
        conversation.append({'role': 'model', 'parts': [raw]})
        conversation.append({'role': 'user',  'parts': [f"Tool '{tool_name}' returned:\n{result}\n\nWhat is your next action? Reply with JSON only."]})
    else:
        print(f'   Unknown tool: {tool_name}')
        conversation.append({'role': 'model', 'parts': [raw]})
        conversation.append({'role': 'user',  'parts': [f"Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}. Try again."]})

print('\nKEY TAKEAWAY:')
print('   The LLM\'s brain is the same as Demo 1A — but now it lives inside')
print('   a loop that lets it CALL REAL FUNCTIONS. That loop is what makes it an AGENT.')